# Hashes, HMAC, and Password Hashing

These examples use the running scenario of St. Isidore Hospital. They are teaching examples: understand the mechanism, then prefer well-reviewed libraries and current protocols in production.

## Goal

Hashes detect change, HMAC authenticates a message with a shared secret, and password hashing deliberately slows attackers down.

In [ ]:
import hashlib

report = b"Radiology report for patient 2048: no fracture."
digest = hashlib.sha256(report).hexdigest()  # Hash the exact byte sequence.
print(digest)

changed = b"Radiology report for patient 2048: fracture."
print(hashlib.sha256(changed).hexdigest())  # A small content change gives a different digest.

In [ ]:
import hmac
from secrets import token_bytes

mac_key = token_bytes(32)  # Shared secret used only by sender and receiver.
message = b"LAB|patient=2048|test=HbA1c|value=6.8"
tag = hmac.new(mac_key, message, hashlib.sha256).hexdigest()  # Authenticate message bytes.
print(tag)

received = b"LAB|patient=2048|test=HbA1c|value=8.8"
received_tag = hmac.new(mac_key, received, hashlib.sha256).hexdigest()  # Recompute on received data.
print("Valid?", hmac.compare_digest(tag, received_tag))  # Constant-time comparison avoids leaks.

In [ ]:
from cryptography.hazmat.primitives.kdf.scrypt import Scrypt
from cryptography.hazmat.backends import default_backend
import os

password = b"CorrectHorseBatteryStaple!"
salt = os.urandom(16)  # Unique salt prevents identical passwords from having identical hashes.
kdf = Scrypt(salt=salt, length=32, n=2**14, r=8, p=1, backend=default_backend())  # Slow KDF.
password_hash = kdf.derive(password)  # Store this verifier, not the plaintext password.
print("salt:", salt.hex())
print("stored hash:", password_hash.hex())